## Setup and Imports

In [ ]:
import os
import sys
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from PIL import Image
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import defaultdict, Counter
import warnings
warnings.filterwarnings('ignore')

# Import model definitions
from model_def import ResNetClassifier, load_trained_model

print("✅ All imports successful")
print(f"PyTorch version: {torch.__version__}")
print(f"GPU Available: {torch.cuda.is_available()}")

## Configuration

In [ ]:
# Configuration
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_PATH = 'models/best_stroke.pth'
TEST_DATA_DIR = 'data/test'
CLASS_NAMES = ['Hemorrhagic', 'Ischemic', 'NoStroke']
IMAGE_SIZE = 224

print(f"Device: {DEVICE}")
print(f"Model: {MODEL_PATH}")
print(f"Test data: {TEST_DATA_DIR}")
print(f"Classes: {CLASS_NAMES}")

## Load Model

In [ ]:
# Load the trained model
model = load_trained_model(MODEL_PATH, num_classes=len(CLASS_NAMES), device=DEVICE)
print(f"✅ Model loaded successfully on {DEVICE}")
model.eval()

## Image Preprocessing

In [ ]:
def preprocess_image(image_path):
    """Convert image to model input tensor"""
    transform = transforms.Compose([
        transforms.Grayscale(num_output_channels=1),
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5], std=[0.5])
    ])
    
    img = Image.open(image_path)
    return transform(img).unsqueeze(0)

def get_patient_id(filename):
    """Extract patient ID from filename (e.g., '10003_7.jpg' -> '10003')"""
    return filename.split('_')[0]

print("✅ Preprocessing functions defined")

## Grad-CAM Implementation

In [ ]:
class GradCAM:
    """Gradient-weighted Class Activation Mapping for model interpretability"""
    
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        
        # Register hooks
        self.target_layer.register_forward_hook(self._save_activation)
        self.target_layer.register_backward_hook(self._save_gradient)
    
    def _save_activation(self, module, input, output):
        """Save activation during forward pass"""
        self.activations = output.detach()
    
    def _save_gradient(self, module, grad_input, grad_output):
        """Save gradient during backward pass"""
        self.gradients = grad_output[0].detach()
    
    def generate_cam(self, input_tensor, class_idx):
        """Generate Class Activation Map"""
        # Forward pass
        self.model.eval()
        output = self.model(input_tensor)
        
        # Backward pass
        self.model.zero_grad()
        target_score = output[0, class_idx]
        target_score.backward()
        
        # Calculate CAM
        gradients = self.gradients[0]
        activations = self.activations[0]
        
        # Weight activations
        weights = gradients.mean(dim=(1, 2))
        cam = torch.zeros_like(activations[0])
        
        for i, w in enumerate(weights):
            cam += w * activations[i]
        
        # ReLU
        cam = torch.nn.functional.relu(cam)
        
        # Normalize
        if cam.max() > 0:
            cam = cam / cam.max()
        
        return cam.cpu().numpy()

def generate_gradcam(model, image_tensor, class_idx, device=DEVICE):
    """Generate Grad-CAM heatmap"""
    try:
        image_tensor = image_tensor.to(device)
        
        # Get the last convolutional layer
        if hasattr(model, 'model'):
            target_layer = model.model.layer4[1].conv2
        else:
            target_layer = model.layer4[1].conv2
        
        # Create GradCAM object
        gradcam = GradCAM(model, target_layer)
        
        # Generate CAM
        cam = gradcam.generate_cam(image_tensor, class_idx)
        
        # Resize CAM to original image size
        cam_resized = cv2.resize(cam, (IMAGE_SIZE, IMAGE_SIZE))
        
        return cam_resized
    except Exception as e:
        print(f"Could not generate Grad-CAM: {str(e)}")
        return None

print("✅ Grad-CAM class defined")

## Prediction Function

In [ ]:
def predict_image(model, image_path, device=DEVICE):
    """Make prediction on single image"""
    img_tensor = preprocess_image(image_path)
    img_tensor = img_tensor.to(device)
    
    with torch.no_grad():
        outputs = model(img_tensor)
        probabilities = F.softmax(outputs, dim=1)
        confidence, predicted_idx = torch.max(probabilities, 1)
    
    return {
        'predicted_class': CLASS_NAMES[predicted_idx.item()],
        'class_idx': predicted_idx.item(),
        'confidence': confidence.item(),
        'probabilities': {CLASS_NAMES[i]: probabilities[0][i].item() for i in range(len(CLASS_NAMES))}
    }

print("✅ Prediction function defined")

## Load Test Images

In [ ]:
# Load all test images with their true labels
test_images = []
true_labels = []
image_paths = []

for class_name in CLASS_NAMES:
    class_dir = os.path.join(TEST_DATA_DIR, class_name)
    if os.path.exists(class_dir):
        for img_file in os.listdir(class_dir):
            if img_file.endswith(('.jpg', '.jpeg', '.png')):
                image_paths.append(os.path.join(class_dir, img_file))
                true_labels.append(class_name)
                test_images.append(img_file)

print(f"✅ Loaded {len(test_images)} test images")
print(f"   Hemorrhagic: {true_labels.count('Hemorrhagic')}")
print(f"   Ischemic: {true_labels.count('Ischemic')}")
print(f"   NoStroke: {true_labels.count('NoStroke')}")

## Batch Predictions

In [ ]:
# Make predictions on all test images
predictions = []
patient_ids = []

print("Making predictions...")
for i, image_path in enumerate(image_paths):
    if (i + 1) % 50 == 0:
        print(f"  Processed {i + 1}/{len(image_paths)}")
    
    result = predict_image(model, image_path, device=DEVICE)
    predictions.append(result)
    patient_ids.append(get_patient_id(test_images[i]))

print(f"✅ Completed predictions on {len(predictions)} images")

## Image-Level Results

In [ ]:
# Create results dataframe
results_df = pd.DataFrame({
    'image': test_images,
    'patient_id': patient_ids,
    'true_class': true_labels,
    'predicted_class': [p['predicted_class'] for p in predictions],
    'confidence': [p['confidence'] for p in predictions],
    'correct': [p['predicted_class'] == true_labels[i] for i, p in enumerate(predictions)]
})

# Calculate metrics
accuracy = results_df['correct'].sum() / len(results_df)
print(f"\n📊 IMAGE-LEVEL METRICS")
print(f"{'=' * 50}")
print(f"Total Images: {len(results_df)}")
print(f"Correct Predictions: {results_df['correct'].sum()}")
print(f"Accuracy: {accuracy:.2%}")
print(f"\nAverage Confidence: {results_df['confidence'].mean():.2%}")
print(f"Min Confidence: {results_df['confidence'].min():.2%}")
print(f"Max Confidence: {results_df['confidence'].max():.2%}")

print(f"\nPer-Class Accuracy:")
for class_name in CLASS_NAMES:
    class_acc = results_df[results_df['true_class'] == class_name]['correct'].sum() / len(results_df[results_df['true_class'] == class_name])
    count = len(results_df[results_df['true_class'] == class_name])
    print(f"  {class_name}: {class_acc:.2%} ({count} images)")

## Patient-Level Aggregation

In [ ]:
# Aggregate predictions by patient ID (majority voting)
patient_data = defaultdict(list)

for i, row in results_df.iterrows():
    patient_id = row['patient_id']
    patient_data[patient_id].append({
        'image': row['image'],
        'true_class': row['true_class'],
        'predicted_class': row['predicted_class'],
        'confidence': row['confidence'],
        'correct': row['correct']
    })

# Compute patient-level predictions
patient_results = []

for patient_id, images in patient_data.items():
    # Get ground truth (should be same for all images of same patient)
    true_class = images[0]['true_class']
    
    # Majority voting
    predictions_list = [img['predicted_class'] for img in images]
    vote_counter = Counter(predictions_list)
    predicted_class = vote_counter.most_common(1)[0][0]
    
    # Average confidence
    avg_confidence = np.mean([img['confidence'] for img in images])
    
    # Correct prediction
    correct = predicted_class == true_class
    
    patient_results.append({
        'patient_id': patient_id,
        'num_slices': len(images),
        'true_class': true_class,
        'predicted_class': predicted_class,
        'avg_confidence': avg_confidence,
        'correct': correct,
        'class_votes': dict(vote_counter)
    })

patient_df = pd.DataFrame(patient_results)
print(f"✅ Aggregated {len(patient_df)} patients")

## Patient-Level Metrics

In [ ]:
# Calculate patient-level metrics
patient_accuracy = patient_df['correct'].sum() / len(patient_df)

print(f"\n👥 PATIENT-LEVEL METRICS")
print(f"{'=' * 50}")
print(f"Total Patients: {len(patient_df)}")
print(f"Correct Patients: {patient_df['correct'].sum()}")
print(f"Patient-Level Accuracy: {patient_accuracy:.2%}")
print(f"\nSlices per Patient:")
print(f"  Min: {patient_df['num_slices'].min()}")
print(f"  Max: {patient_df['num_slices'].max()}")
print(f"  Mean: {patient_df['num_slices'].mean():.1f}")
print(f"\nAverage Confidence (Patient-level):")
print(f"  Mean: {patient_df['avg_confidence'].mean():.2%}")
print(f"  Min: {patient_df['avg_confidence'].min():.2%}")
print(f"  Max: {patient_df['avg_confidence'].max():.2%}")

print(f"\nPer-Class Patient Accuracy:")
for class_name in CLASS_NAMES:
    class_df = patient_df[patient_df['true_class'] == class_name]
    if len(class_df) > 0:
        class_acc = class_df['correct'].sum() / len(class_df)
        print(f"  {class_name}: {class_acc:.2%} ({len(class_df)} patients)")

## Visualization - Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix

# Image-level confusion matrix
cm_images = confusion_matrix(results_df['true_class'], results_df['predicted_class'], labels=CLASS_NAMES)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Image-level
sns.heatmap(cm_images, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[0])
axes[0].set_title('Image-Level Confusion Matrix', fontsize=12, fontweight='bold')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

# Patient-level confusion matrix
cm_patients = confusion_matrix(patient_df['true_class'], patient_df['predicted_class'], labels=CLASS_NAMES)
sns.heatmap(cm_patients, annot=True, fmt='d', cmap='Greens', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[1])
axes[1].set_title('Patient-Level Confusion Matrix', fontsize=12, fontweight='bold')
axes[1].set_ylabel('True Label')
axes[1].set_xlabel('Predicted Label')

plt.tight_layout()
plt.show()

print("✅ Confusion matrices displayed")

## Grad-CAM Visualization - Sample Images

In [ ]:
def visualize_gradcam(image_path, true_class, pred_result):
    """Visualize image with Grad-CAM heatmap"""
    # Load and preprocess image
    img_pil = Image.open(image_path).convert('L')
    img_np = np.array(img_pil)
    img_tensor = preprocess_image(image_path).to(DEVICE)
    
    # Generate Grad-CAM
    cam = generate_gradcam(model, img_tensor, pred_result['class_idx'], device=DEVICE)
    
    if cam is None:
        return None
    
    # Create visualization
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # Original image
    axes[0].imshow(img_np, cmap='gray')
    axes[0].set_title('Original Image', fontweight='bold')
    axes[0].axis('off')
    
    # Grad-CAM heatmap
    axes[1].imshow(img_np, cmap='gray')
    axes[1].imshow(cam, cmap='hot', alpha=0.5)
    axes[1].set_title('Grad-CAM Heatmap', fontweight='bold')
    axes[1].axis('off')
    
    # Probability distribution
    probs = pred_result['probabilities']
    colors = ['green' if CLASS_NAMES[i] == pred_result['predicted_class'] else 'lightgray' for i in range(len(CLASS_NAMES))]
    axes[2].barh(CLASS_NAMES, [probs[cls] for cls in CLASS_NAMES], color=colors)
    axes[2].set_xlabel('Probability')
    axes[2].set_title('Prediction Probabilities', fontweight='bold')
    axes[2].set_xlim([0, 1])
    
    plt.suptitle(f'True: {true_class} | Predicted: {pred_result["predicted_class"]} ({pred_result["confidence"]:.1%})', 
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    
    return fig

print("✅ Grad-CAM visualization function defined")

## Example: Grad-CAM for Correct Predictions

In [ ]:
# Select one correct prediction per class
print("Visualizing Grad-CAM for correct predictions...\n")

for class_name in CLASS_NAMES:
    correct_preds = results_df[(results_df['true_class'] == class_name) & (results_df['correct'] == True)]
    
    if len(correct_preds) > 0:
        # Select highest confidence prediction
        best_idx = correct_preds['confidence'].idxmax()
        row = results_df.loc[best_idx]
        
        image_path = os.path.join(TEST_DATA_DIR, row['true_class'], row['image'])
        pred_result = predictions[best_idx]
        
        print(f"\n{class_name.upper()} - Correct Prediction")
        print(f"File: {row['image']}")
        print(f"Confidence: {pred_result['confidence']:.2%}")
        
        fig = visualize_gradcam(image_path, row['true_class'], pred_result)
        if fig:
            plt.show()
    else:
        print(f"\n⚠️ No correct {class_name} predictions found")

## Example: Grad-CAM for Misclassifications

In [ ]:
# Select one misclassification per class
print("Visualizing Grad-CAM for misclassifications...\n")

for class_name in CLASS_NAMES:
    wrong_preds = results_df[(results_df['true_class'] == class_name) & (results_df['correct'] == False)]
    
    if len(wrong_preds) > 0:
        # Select highest confidence misclassification
        worst_idx = wrong_preds['confidence'].idxmax()
        row = results_df.loc[worst_idx]
        
        image_path = os.path.join(TEST_DATA_DIR, row['true_class'], row['image'])
        pred_result = predictions[worst_idx]
        
        print(f"\n{class_name.upper()} - Misclassification")
        print(f"File: {row['image']}")
        print(f"True: {row['true_class']}, Predicted: {row['predicted_class']} ({pred_result['confidence']:.2%})")
        
        fig = visualize_gradcam(image_path, row['true_class'], pred_result)
        if fig:
            plt.show()
    else:
        print(f"\n✅ No {class_name} misclassifications - all correct!")

## Summary Statistics

In [ ]:
print("\n" + "="*60)
print("COMPREHENSIVE ANALYSIS SUMMARY")
print("="*60)

print("\n📊 IMAGE-LEVEL ANALYSIS")
print(f"  Total Images: {len(results_df)}")
print(f"  Accuracy: {(results_df['correct'].sum() / len(results_df)):.2%}")
print(f"  Mean Confidence: {results_df['confidence'].mean():.2%}")

print("\n👥 PATIENT-LEVEL ANALYSIS")
print(f"  Total Patients: {len(patient_df)}")
print(f"  Accuracy: {(patient_df['correct'].sum() / len(patient_df)):.2%}")
print(f"  Mean Patient Confidence: {patient_df['avg_confidence'].mean():.2%}")

print("\n🎯 CLASS DISTRIBUTION (Test Set)")
for class_name in CLASS_NAMES:
    count = len(results_df[results_df['true_class'] == class_name])
    pct = count / len(results_df) * 100
    print(f"  {class_name}: {count} images ({pct:.1f}%)")

print("\n✅ Analysis complete!")

## Export Results

In [ ]:
# Export image-level results
results_df.to_csv('results/image_level_predictions.csv', index=False)
print(f"✅ Exported image-level predictions to results/image_level_predictions.csv")

# Export patient-level results
patient_df.to_csv('results/patient_level_predictions.csv', index=False)
print(f"✅ Exported patient-level predictions to results/patient_level_predictions.csv")

print(f"\n📁 Results saved successfully!")